# Notebook 01 - AI Search Por Usuario

Crea AI Search en el RG del participante y carga un indice vectorial de conocimiento.

### Descripcion de la celda 1
Carga configuracion, valida variables y prepara el contexto.

In [40]:
# Celda 2: ejecuta este paso del notebook y prepara resultados para el siguiente.

import json
import os
import shutil
import subprocess
import requests
from pathlib import Path
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

def resolve_workdir() -> Path:
    cwd = Path.cwd()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for c in candidates:
        if (c / 'config.env').exists():
            return c
    return cwd

def resolve_az_cli() -> str:
    for candidate in ('az', 'az.cmd', 'az.exe'):
        found = shutil.which(candidate)
        if found:
            return found
    raise RuntimeError(
        'No se encontro Azure CLI en el PATH del kernel. '
        'En Windows instala Azure CLI y reinicia VS Code/Jupyter.'
    )

WORKDIR = resolve_workdir()
AZ_CLI = resolve_az_cli()

def load_env_file(path: Path):
    env_map = {}
    if not path.exists():
        return env_map
    for raw in path.read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        k, v = line.split('=', 1)
        env_map[k.strip()] = v.strip().strip("\"").strip("'")
    return env_map

env_file = load_env_file(WORKDIR / 'config.env')
cfg = {}
cfg_path = WORKDIR / 'workshop_config.json'
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text(encoding='utf-8'))

def env(name, default=None, required=False):
    v = os.getenv(name, env_file.get(name, default))
    if required and (v is None or str(v).strip() == ''):
        raise RuntimeError(f'Falta variable requerida: {name}')
    return v

SUBSCRIPTION_ID = env('AZURE_SUBSCRIPTION_ID', cfg.get('azure_subscription_id'), required=True)
LOCATION = env('AZURE_LOCATION', cfg.get('azure_location', 'eastus'))
USER_ALIAS = env('USER_ALIAS', cfg.get('user_alias', 'user01')).strip()
USER_RG_NAME = env('USER_RG_NAME', cfg.get('user_rg_name', f'rg-foundry-demo-{USER_ALIAS}'))
RESOURCE_PREFIX = env('RESOURCE_PREFIX', cfg.get('resource_prefix', f'contoso-{USER_ALIAS}'))
SEARCH_SERVICE_NAME = f"{USER_ALIAS}srch".replace('-', '')[:48]
SEARCH_INDEX_NAME = f"{RESOURCE_PREFIX}-kb".lower()[:128]

cfg = {
    **cfg,
    'azure_openai_endpoint': env('AZURE_OPENAI_ENDPOINT', cfg.get('azure_openai_endpoint'), required=True),
    'api_version': env('AZURE_OPENAI_API_VERSION', env('API_VERSION', cfg.get('api_version', '2025-01-01-preview'))),
    'embedding_model_deployment': env('EMBEDDING_MODEL_DEPLOYMENT', cfg.get('embedding_model_deployment', ''))
}

def run(cmd):
    final_cmd = cmd.copy()
    if final_cmd and final_cmd[0] == 'az':
        final_cmd[0] = AZ_CLI
    p = subprocess.run(final_cmd, capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(p.stderr or p.stdout)
    return p.stdout.strip()

run(['az', 'account', 'set', '--subscription', SUBSCRIPTION_ID])
print('Workdir:', WORKDIR)
print('Azure CLI:', AZ_CLI)
print('Search service:', SEARCH_SERVICE_NAME)
print('Index:', SEARCH_INDEX_NAME)
print('Resource group:', USER_RG_NAME)

Workdir: /Users/williamtalero/Documents/Proyectos/Microsoft/WorkShop - Agentes
Azure CLI: /opt/homebrew/bin/az
Search service: user29srch
Index: contoso-user29-kb
Resource group: rg-foundry-demo-user29


### Descripcion de la celda 2
Crea o valida Azure AI Search y deja el indice listo.

In [41]:
# Asegura que exista el RG por usuario para este laboratorio
run([
    'az', 'group', 'create',
    '--name', USER_RG_NAME,
    '--location', LOCATION,
    '-o', 'none'
])

probe = subprocess.run([
    AZ_CLI, 'search', 'service', 'show',
    '--name', SEARCH_SERVICE_NAME,
    '--resource-group', USER_RG_NAME
], capture_output=True, text=True)

if probe.returncode != 0:
    run([
        'az', 'search', 'service', 'create',
        '--name', SEARCH_SERVICE_NAME,
        '--resource-group', USER_RG_NAME,
        '--location', LOCATION,
        '--sku', 'basic'
    ])
    print('Search creado')
else:
    print('Search ya existia')

search_admin_key = run([
    'az', 'search', 'admin-key', 'show',
    '--service-name', SEARCH_SERVICE_NAME,
    '--resource-group', USER_RG_NAME,
    '--query', 'primaryKey', '-o', 'tsv'
])
search_endpoint = f"https://{SEARCH_SERVICE_NAME}.search.windows.net"
search_api_version = '2024-07-01'

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, 'https://cognitiveservices.azure.com/.default')
aoai = AzureOpenAI(
    azure_endpoint=cfg['azure_openai_endpoint'],
    azure_ad_token_provider=token_provider,
    api_version=cfg['api_version']
)

embedding_deployment = cfg.get('embedding_model_deployment', '').strip()
USE_VECTOR_SEARCH = True
VECTOR_DIMENSIONS = None

def _find_embedding_deployments_from_endpoint(endpoint: str):
    normalized = endpoint.rstrip('/')
    account_tsv = run([
        'az', 'cognitiveservices', 'account', 'list',
        '--query', f"[?properties.endpoint=='{normalized}'].[name,resourceGroup] | [0]",
        '-o', 'tsv'
    ])
    if not account_tsv:
        return []
    account_name, account_rg = account_tsv.split('\t')
    deployments_tsv = run([
        'az', 'cognitiveservices', 'account', 'deployment', 'list',
        '--name', account_name,
        '--resource-group', account_rg,
        '--query', "[?contains(properties.model.name, 'embedding')].name",
        '-o', 'tsv'
    ])
    return [d.strip() for d in deployments_tsv.splitlines() if d.strip()]

if embedding_deployment:
    try:
        probe_emb = aoai.embeddings.create(model=embedding_deployment, input='ping')
        VECTOR_DIMENSIONS = len(probe_emb.data[0].embedding)
    except Exception as ex:
        if 'DeploymentNotFound' not in str(ex):
            raise
        candidates = _find_embedding_deployments_from_endpoint(cfg['azure_openai_endpoint'])
        if candidates:
            embedding_deployment = candidates[0]
            cfg['embedding_model_deployment'] = embedding_deployment
            probe_emb = aoai.embeddings.create(model=embedding_deployment, input='ping')
            VECTOR_DIMENSIONS = len(probe_emb.data[0].embedding)
            print(f"Embedding deployment configurado no existe. Usando fallback: {embedding_deployment}")
        else:
            USE_VECTOR_SEARCH = False
            embedding_deployment = ''
            print('No hay deployment de embeddings disponible. Se usara indice textual.')
else:
    USE_VECTOR_SEARCH = False
    print('No se definio embedding_model_deployment. Se usara indice textual.')

print('Endpoint:', search_endpoint)
print('Vector search habilitado:', USE_VECTOR_SEARCH)
if USE_VECTOR_SEARCH:
    print('Embedding deployment:', embedding_deployment)
    print('Vector dimensions:', VECTOR_DIMENSIONS)

Search creado
Endpoint: https://user29srch.search.windows.net
Vector search habilitado: True
Embedding deployment: text-embedding-3-small
Vector dimensions: 1536


### Descripcion de la celda 3
Crea o valida Azure AI Search y deja el indice listo.

In [42]:
# Celda 4: ejecuta este paso del notebook y prepara resultados para el siguiente.

fields = [
  {'name': 'id', 'type': 'Edm.String', 'key': True, 'searchable': False, 'filterable': True, 'retrievable': True},
  {'name': 'source', 'type': 'Edm.String', 'searchable': True, 'filterable': True, 'retrievable': True},
  {'name': 'content', 'type': 'Edm.String', 'searchable': True, 'retrievable': True}
 ]

index_schema = {'name': SEARCH_INDEX_NAME, 'fields': fields}

if USE_VECTOR_SEARCH:
  index_schema['fields'].append({
      'name': 'contentVector',
      'type': 'Collection(Edm.Single)',
      'searchable': True,
      'retrievable': False,
      'dimensions': VECTOR_DIMENSIONS,
      'vectorSearchProfile': 'vprofile'
  })
  index_schema['vectorSearch'] = {
      'algorithms': [{'name': 'halgo', 'kind': 'hnsw'}],
      'profiles': [{'name': 'vprofile', 'algorithm': 'halgo'}]
  }

resp = requests.put(
    f"{search_endpoint}/indexes/{SEARCH_INDEX_NAME}?api-version={search_api_version}",
    headers={'Content-Type': 'application/json', 'api-key': search_admin_key},
    json=index_schema
)
if resp.status_code not in (200, 201, 204):
    raise RuntimeError(resp.text or f'Error creando indice. HTTP {resp.status_code}')
print('Indice listo. Modo vector:', USE_VECTOR_SEARCH)

Indice listo. Modo vector: True


### Descripcion de la celda 4
Crea o valida Azure AI Search y deja el indice listo.

In [43]:
# Celda 5: ejecuta este paso del notebook y prepara resultados para el siguiente.

def chunk_text(text, size=1200, overlap=150):
    out = []
    i = 0
    while i < len(text):
        out.append(text[i:i+size])
        i += max(1, size-overlap)
    return out

docs = [
    ('faq', (WORKDIR / 'data' / 'faq_contoso.md').read_text(encoding='utf-8')),
    ('politicas', (WORKDIR / 'data' / 'politicas_credito.md').read_text(encoding='utf-8'))
]

items = []
for source, text in docs:
    for idx, part in enumerate(chunk_text(text)):
        item = {
            'id': f'{source}-{idx}',
            'source': source,
            'content': part
        }
        if USE_VECTOR_SEARCH:
            emb = aoai.embeddings.create(model=embedding_deployment, input=part)
            item['contentVector'] = emb.data[0].embedding
        items.append(item)

payload = {'value': [{'@search.action': 'upload', **d} for d in items]}
ru = requests.post(
    f"{search_endpoint}/indexes/{SEARCH_INDEX_NAME}/docs/index?api-version={search_api_version}",
    headers={'Content-Type': 'application/json', 'api-key': search_admin_key},
    json=payload
)
if ru.status_code not in (200, 201):
    raise RuntimeError(ru.text)
print('Chunks cargados:', len(items))

Chunks cargados: 10


### Descripcion de la celda 5
Actualiza y guarda el estado para los siguientes notebooks.

In [44]:
# Celda 6: ejecuta este paso del notebook y prepara resultados para el siguiente.

if 'search_endpoint' not in globals() or 'search_admin_key' not in globals():
    raise RuntimeError('AI Search no inicializado. Ejecuta celdas previas y corrige errores.')
if 'items' not in globals() or not items:
    raise RuntimeError('No hay documentos indexados. Ejecuta la carga de chunks antes de finalizar.')

AI_SEARCH_CONFIG = {
    'service_name': SEARCH_SERVICE_NAME,
    'index_name': SEARCH_INDEX_NAME,
    'endpoint': search_endpoint,
    'admin_key': search_admin_key,
    'api_version': search_api_version,
    'use_vector_search': USE_VECTOR_SEARCH
}
if USE_VECTOR_SEARCH:
    AI_SEARCH_CONFIG['vector_dimensions'] = VECTOR_DIMENSIONS
    AI_SEARCH_CONFIG['embedding_model_deployment'] = embedding_deployment

state = {}
config_path = WORKDIR / 'workshop_config.json'
if config_path.exists():
    state = json.loads(config_path.read_text(encoding='utf-8'))

state.update({
    'azure_subscription_id': SUBSCRIPTION_ID,
    'azure_location': LOCATION,
    'user_alias': USER_ALIAS,
    'user_rg_name': USER_RG_NAME,
    'resource_prefix': RESOURCE_PREFIX,
    'azure_openai_endpoint': cfg['azure_openai_endpoint'],
    'api_version': cfg['api_version'],
    'embedding_model_deployment': embedding_deployment or cfg.get('embedding_model_deployment', ''),
    'ai_search': AI_SEARCH_CONFIG,
    'ai_search_ready': True
})
config_path.write_text(json.dumps(state, indent=2), encoding='utf-8')

print('OK AI Search configurado')
print('Estado actualizado:', config_path)
print(json.dumps(AI_SEARCH_CONFIG, indent=2))

OK AI Search configurado
Estado actualizado: /Users/williamtalero/Documents/Proyectos/Microsoft/WorkShop - Agentes/workshop_config.json
{
  "service_name": "user29srch",
  "index_name": "contoso-user29-kb",
  "endpoint": "https://user29srch.search.windows.net",
  "admin_key": "***REDACTED***",
  "api_version": "2024-07-01",
  "use_vector_search": true,
  "vector_dimensions": 1536,
  "embedding_model_deployment": "text-embedding-3-small"
}
